In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2","PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2","PLATE5_T0","PLATE5_T1","PLATE5_T2"]

# --- ADDED: Define where you want to save the CSV ---
OUTPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")

TREATMENT_COL = "Treatment"
NUM_CHANNELS = 5
FEATS_PER_CH = 1280

all_plates_data = []

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT,"features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT,"metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_median = np.median(all_cells_in_well, axis=0)
        
        row = {"Plate": plate_id, "Well_ID": well_id, "Treatment": well_to_treatment[well_id]}
        for idx, val in enumerate(well_median):
            row[idx] = val
        all_plates_data.append(row)

# Combine everything into one giant DataFrame
wells_df = pd.DataFrame(all_plates_data)
print(f"\nAggregation complete. Total Wells: {len(wells_df)}")

# ==========================================
# 2. SAVE TO CSV
# ==========================================
# index=False prevents pandas from writing a column for the row numbers
wells_df.to_csv(OUTPUT_CSV, index=False)
print(f"Successfully saved to: {OUTPUT_CSV}")

In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# Define the channel subsets (same as before)
feature_cols = [i for i in range(NUM_CHANNELS * FEATS_PER_CH)]
plot_configs = [
    {"name": "All Channels", "indices": feature_cols},
    {"name": "Channel 1", "indices": list(range(0, 1280))},
    {"name": "Channel 2", "indices": list(range(1280, 2560))},
    {"name": "Channel 3", "indices": list(range(2560, 3840))},
    {"name": "Channel 4", "indices": list(range(3840, 5120))},
    {"name": "Channel 5", "indices": list(range(5120, 6400))}
]

# LOOP THROUGH EACH PLATE INDIVIDUALLY
for plate_id in wells_df['Plate'].unique():
    plate_subset = wells_df[wells_df['Plate'] == plate_id].copy()
    
    fig, axes = plt.subplots(2, 3, figsize=(26, 18))
    axes = axes.flatten()

    for idx, config in enumerate(plot_configs):
        ax = axes[idx]
        
        # 1. Prepare & Scale Data
        X_subset = plate_subset[config['indices']].values
        X_scaled = StandardScaler().fit_transform(X_subset)
        
        # 2. Run UMAP
        reducer = umap.UMAP(n_neighbors=min(15, len(plate_subset)-1), min_dist=0.1, random_state=42)
        embedding = reducer.fit_transform(X_scaled)
        
        # 3. Plotting logic (same as your original code)
        is_control = plate_subset[TREATMENT_COL] == "no_sgRNA"
        
        # Draw Control vs Mutants
        ax.scatter(embedding[is_control, 0], embedding[is_control, 1], 
                   c='red', marker='x', s=120, label='Control', zorder=4)
        ax.scatter(embedding[~is_control, 0], embedding[~is_control, 1], 
                   c='black', marker='o', s=50, alpha=0.5, label='Mutants', zorder=3)

        # 4. Text Labels (using subset data)
        texts = []
        for i, row in enumerate(plate_subset.itertuples()):
            color = 'red' if row.Treatment == "no_sgRNA" else 'black'
            label = row.Well_ID if row.Treatment == "no_sgRNA" else row.Treatment
            texts.append(ax.text(embedding[i, 0], embedding[i, 1], label, fontsize=7, color=color))

        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='silver', lw=0.5))
        ax.set_title(f"{plate_id} - {config['name']}")

    plt.suptitle(f"UMAP Analysis: {plate_id}", fontsize=22, y=1.02)
    plt.tight_layout()
    # plt.savefig(f"umap_{plate_id}.png")
    plt.show()

In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text
import matplotlib.cm as cm
import numpy as np

# --- 1. Dynamic Color Setup ---
plate_list = sorted(wells_df['Plate'].unique())
num_plates = len(plate_list)

# Use a qualitative colormap (Set1, Set3, or Dark2 are good for distinct categories)
# 'tab10' or 'tab20' are excellent for up to 10 or 20 distinct categories
cmap = plt.get_cmap('tab10') 
plate_colors = [cmap(i) for i in np.linspace(0, 1, num_plates)]
plate_color_map = {plate: plate_colors[i] for i, plate in enumerate(plate_list)}

# --- 2. Configuration ---
feature_cols = [i for i in range(NUM_CHANNELS * FEATS_PER_CH)]
plot_configs = [
    {"name": "All Channels", "indices": feature_cols},
    {"name": "Channel 1", "indices": list(range(0, 1280))},
    {"name": "Channel 2", "indices": list(range(1280, 2560))},
    {"name": "Channel 3", "indices": list(range(2560, 3840))},
    {"name": "Channel 4", "indices": list(range(3840, 5120))},
    {"name": "Channel 5", "indices": list(range(5120, 6400))}
]

fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    print(f"Running UMAP for {config['name']}...")
    
    X_subset = wells_df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X_subset)
    
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    # 3. Plot Plate by Plate
    for plate in plate_list:
        mask = wells_df['Plate'] == plate
        plate_embed = embedding[mask]
        plate_meta = wells_df[mask]
        
        is_ctrl = plate_meta[TREATMENT_COL] == "no_sgRNA"
        
        # Plot Mutants (Filled Circles)
        ax.scatter(plate_embed[~is_ctrl, 0], plate_embed[~is_ctrl, 1], 
                   color=plate_color_map[plate], marker='o', s=60, alpha=0.6, 
                   label=f"{plate} (Mutant)")
        
        # Plot Controls (Open Squares)
        ax.scatter(plate_embed[is_ctrl, 0], plate_embed[is_ctrl, 1], 
                   edgecolors=plate_color_map[plate], facecolors='none', 
                   marker='s', s=130, linewidths=2, label=f"{plate} (Control)")

        # 4. Add labels
        for i, row in enumerate(plate_meta.itertuples()):
            label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
            texts.append(ax.text(plate_embed[i, 0], plate_embed[i, 1], label, 
                                 fontsize=6, color=plate_color_map[plate]))

    # Formatting
    ax.set_title(f"{config['name']}", fontsize=16, fontweight='bold')
    
    # Move legend to the side if it gets too crowded with 6+ plates
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), markerscale=1, fontsize=8, ncol=1)
    
    # Adjust text to prevent overlapping labels
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("Combined UMAP: Multi-Plate Analysis", fontsize=24, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2","PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2","PLATE5_T0","PLATE5_T1","PLATE5_T2"]

# --- ADDED: Define where you want to save the CSV ---
OUTPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")

TREATMENT_COL = "Treatment"
NUM_CHANNELS = 5
FEATS_PER_CH = 1280

all_plates_data = []

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT,"features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT,"metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_median = np.median(all_cells_in_well, axis=0)
        
        row = {"Plate": plate_id, "Well_ID": well_id, "Treatment": well_to_treatment[well_id]}
        for idx, val in enumerate(well_median):
            row[idx] = val
        all_plates_data.append(row)

# Combine everything into one giant DataFrame
wells_df = pd.DataFrame(all_plates_data)
print(f"\nAggregation complete. Total Wells: {len(wells_df)}")

# ==========================================
# 2. SAVE TO CSV
# ==========================================
# index=False prevents pandas from writing a column for the row numbers
wells_df.to_csv(OUTPUT_CSV, index=False)
print(f"Successfully saved to: {OUTPUT_CSV}")

In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text
import matplotlib.cm as cm
import numpy as np

# --- 1. Dynamic Color Setup ---
plate_list = sorted(wells_df['Plate'].unique())
num_plates = len(plate_list)

# Use a qualitative colormap (Set1, Set3, or Dark2 are good for distinct categories)
# 'tab10' or 'tab20' are excellent for up to 10 or 20 distinct categories
cmap = plt.get_cmap('tab10') 
plate_colors = [cmap(i) for i in np.linspace(0, 1, num_plates)]
plate_color_map = {plate: plate_colors[i] for i, plate in enumerate(plate_list)}

# --- 2. Configuration ---
feature_cols = [i for i in range(NUM_CHANNELS * FEATS_PER_CH)]
plot_configs = [
    {"name": "All Channels", "indices": feature_cols},
    {"name": "Channel 1", "indices": list(range(0, 1280))},
    {"name": "Channel 2", "indices": list(range(1280, 2560))},
    {"name": "Channel 3", "indices": list(range(2560, 3840))},
    {"name": "Channel 4", "indices": list(range(3840, 5120))},
    {"name": "Channel 5", "indices": list(range(5120, 6400))}
]

fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    print(f"Running UMAP for {config['name']}...")
    
    X_subset = wells_df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X_subset)
    
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    # 3. Plot Plate by Plate
    for plate in plate_list:
        mask = wells_df['Plate'] == plate
        plate_embed = embedding[mask]
        plate_meta = wells_df[mask]
        
        is_ctrl = plate_meta[TREATMENT_COL] == "no_sgRNA"
        
        # Plot Mutants (Filled Circles)
        ax.scatter(plate_embed[~is_ctrl, 0], plate_embed[~is_ctrl, 1], 
                   color=plate_color_map[plate], marker='o', s=60, alpha=0.6, 
                   label=f"{plate} (Mutant)")
        
        # Plot Controls (Open Squares)
        ax.scatter(plate_embed[is_ctrl, 0], plate_embed[is_ctrl, 1], 
                   edgecolors=plate_color_map[plate], facecolors='none', 
                   marker='s', s=130, linewidths=2, label=f"{plate} (Control)")

        # 4. Add labels
        for i, row in enumerate(plate_meta.itertuples()):
            label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
            texts.append(ax.text(plate_embed[i, 0], plate_embed[i, 1], label, 
                                 fontsize=6, color=plate_color_map[plate]))

    # Formatting
    ax.set_title(f"{config['name']}", fontsize=16, fontweight='bold')
    
    # Move legend to the side if it gets too crowded with 6+ plates
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), markerscale=1, fontsize=8, ncol=1)
    
    # Adjust text to prevent overlapping labels
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("Combined UMAP: Multi-Plate Analysis", fontsize=24, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Check if your dataframe and the last embedding still exist
print(wells_df.head())
print(embedding.shape)

In [ ]:
# Run this once to 'pin' the results to your dataframe
# If you already ran UMAP, this loop will 'attach' the math to the rows
for config in plot_configs:
    X_subset = wells_df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X_subset)
    
    # This is the slow part, but once it's done, it's done forever
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Store these so you can change colors 100 times in 1 second
    wells_df[f"umap_{config['name']}_x"] = embedding[:, 0]
    wells_df[f"umap_{config['name']}_y"] = embedding[:, 1]

# SAVE TO DISK so you can restart your PC and keep the math
wells_df.to_pickle("umap_results_backup.pkl")

In [ ]:
# Save to your external drive
wells_df.to_pickle("/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/umap_results_backup.pkl")

print("File saved successfully to the external drive!")

In [ ]:
# Extract Timepoint from Plate string (e.g., 'PLATE1_T0' -> 'T0')
wells_df['Timepoint'] = wells_df['Plate'].apply(lambda x: x.split('_')[-1])

# Create a unique group name for the legend/coloring
wells_df['Color_Group'] = wells_df['Plate'] + "_" + wells_df['Timepoint']
unique_groups = sorted(wells_df['Color_Group'].unique())

# Set up a colormap with enough colors for all combinations
cmap = plt.get_cmap('tab20') # 'tab20' is great for ~20 distinct categories
colors = [cmap(i) for i in np.linspace(0, 1, len(unique_groups))]
color_map = dict(zip(unique_groups, colors))

fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    x_col = f"umap_{config['name']}_x"
    y_col = f"umap_{config['name']}_y"
    
    texts = []
    for group in unique_groups:
        mask = wells_df['Color_Group'] == group
        subset = wells_df[mask]
        
        is_ctrl = subset['Treatment'] == "no_sgRNA" # Adjust if your control name is different
        
        # Plot Mutants (Circles)
        ax.scatter(subset.loc[~is_ctrl, x_col], subset.loc[~is_ctrl, y_col], 
                   color=color_map[group], marker='o', s=60, alpha=0.6, label=f"{group}")
        
        # Plot Controls (Squares)
        ax.scatter(subset.loc[is_ctrl, x_col], subset.loc[is_ctrl, y_col], 
                   edgecolors=color_map[group], facecolors='none', marker='s', s=130, linewidths=2)

    ax.set_title(config['name'], fontsize=16, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), ncol=2, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Use raw string (r'') for Windows paths to handle backslashes correctly
# I've updated the path based on your description
file_path = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/aggregated_wells_data.csv"

# Read only the first 5 rows to see the structure
df_preview = pd.read_csv(file_path, nrows=50)

print(f"File structure for: {file_path}")
print(f"Columns found: {df_preview.columns.tolist()[:10]} ... [and 6393 more]")
display(df_preview.head())

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Setup Paths
PROJECT_ROOT =  "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")

# 2. Load Raw Data
print("Loading raw aggregated data...")
df_raw = pd.read_csv(INPUT_CSV)

# Identify features (0-6399)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_on_raw_data(df, features, anchor_plate='PLATE5_T0', top_n=500):
    # Aggregate Raw Control Medians
    ctrls = df[df['Treatment'] == 'no_sgRNA']
    plate_medians = ctrls.groupby('Plate')[features].median()

    # A. Within-Plate Drift (P5 T0 vs P5 T1)
    # On raw data, this shows how much the machine/environment drifts over time
    within_plate_drift = (plate_medians.loc['PLATE5_T1'] - plate_medians.loc['PLATE5_T0']).abs()

    # B. Across-Plate Drift (P1 T0 vs P5 T0)
    # On raw data, this shows the true technical difference between physical plates
    anchor_t0 = plate_medians.loc['PLATE5_T0']
    other_t0s = [p for p in plate_medians.index if '_T0' in p and p != 'PLATE5_T0']
    
    across_plate_drift = pd.concat([
        (plate_medians.loc[p] - anchor_t0).abs() for p in other_t0s
    ], axis=1).mean(axis=1)

    # C. Total Technical Noise
    total_tech_noise = (within_plate_drift + across_plate_drift) / 2

    # D. Biological Signal (Mutant Impact)
    # How much do mutants deviate from the overall raw control median?
    global_ctrl_median = ctrls[features].median()
    mutant_impact = (df[features] - global_ctrl_median).abs().quantile(0.95)

    # E. Final Quality Score
    # Features where Mutant Impact is high relative to Raw Technical Noise
    quality_score = mutant_impact / (total_tech_noise + 1e-6)

    # Pick top features
    variance_filter = df[features].std() > 0.01
    vetted_list = quality_score[variance_filter].sort_values(ascending=False).head(top_n).index.tolist()
    
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_on_raw_data(df_raw, feature_cols, top_n=500)

# Now that we have the features, we can create the filtered dataframe
df_vetted_raw = df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols]

print(f"Features selected! Best feature index: {vetted_cols[0]}")

In [ ]:
def robust_normalize_final(df, features):
    normalized_list = []
    for plate in df['Plate'].unique():
        plate_df = df[df['Plate'] == plate].copy()
        controls = plate_df[plate_df['Treatment'] == 'nosgrna']
        if not controls.empty:
            median = controls[features].median()
            mad = (controls[features] - median).abs().median()
            plate_df[features] = (plate_df[features] - median) / (mad + 1e-6)
        normalized_list.append(plate_df)
    return pd.concat(normalized_list)

df_final_normalized = robust_normalize_final(df_vetted_raw, vetted_cols)

# Save the final masterpiece
df_final_normalized.to_csv(os.path.join(PROJECT_ROOT, "vetted_biological_profiles.csv"), index=False)
print("Final normalized and vetted profiles saved.")

In [ ]:
import pandas as pd
import numpy as np
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# 1. Load your vetted and normalized data
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_biological_profiles.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. Dynamic Color Setup (Every Plate + Timepoint gets a unique color)
# We use 'tab20' which has 20 distinct colors. If you have more, we cycle them.
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab20') 

# This creates a dictionary where 'PLATE1_T0' has a different color than 'PLATE1_T1'
plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# 3. Define Channel Groups
# We identify which of our 500 vetted features belong to which original channel
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1 ", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. Run the UMAP Grid
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (No features survived vetting)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    # Plot Plate-by-Plate (Distinct colors for PLATE1_T0, PLATE1_T1, etc.)
    for plate in plate_list:
        mask = df['Plate'] == plate
        plate_embed = embedding[mask]
        plate_meta = df[mask]
        
        # Identify controls using your string
        is_ctrl = plate_meta['Treatment'] == "no_sgRNA"
        color = plate_color_map[plate]
        
        # Plot Mutants (Filled Circles)
        ax.scatter(plate_embed[~is_ctrl, 0], plate_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, 
                   label=f"{plate}")
        
        # Plot Controls (Open Squares - larger to stand out)
        ax.scatter(plate_embed[is_ctrl, 0], plate_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # 5. Add Labels for Mutants
        for i, row in enumerate(plate_meta.itertuples()):
            if row.Treatment != "nosgrna":
                # Label format: Target_Timepoint
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(plate_embed[i, 0], plate_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    # Formatting
    ax.set_title(f"{config['name']} ({len(config['indices'])} features)", fontsize=18, fontweight='bold')
    if idx == 0:
        # One legend for the whole figure
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, ncol=1, title="Plate & Timepoint")
    
    # Adjust text to prevent overlapping labels
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis: Vetted & Balanced Features", fontsize=28, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "umap_vetted_colored_by_timepoint.png"), dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

# 2. LOAD RAW DATA
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_global_stability(df, features, top_n=500):
    # Get medians for controls per plate
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # Timepoints to anchor (Across-Plate Consistency)
    timepoints = ['T0', 'T1', 'T2']
    
    # A. Calculate Across-Plate Noise for EACH timepoint
    # (Checking if Plate 1 T0 looks like Plate 5 T0, etc.)
    tp_noises = []
    for tp in timepoints:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std(axis=0)
            tp_noises.append(noise)
    
    global_across_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)

    # B. Within-Plate Drift (ONLY T0 vs T1)
    # Since T2 has biological growth, we only use T0 and T1 to measure technical drift
    if 'PLATE5_T0' in plate_medians.index and 'PLATE5_T1' in plate_medians.index:
        internal_drift = (plate_medians.loc['PLATE5_T1'] - plate_medians.loc['PLATE5_T0']).abs()
    else:
        print("Warning: PLATE5_T0 or T1 missing. Internal drift set to 0.")
        internal_drift = 0

    # C. Combined Technical Penalty
    total_penalty = (global_across_plate_noise + internal_drift) / 2

    # D. Biological Signal (Mutant Impact)
    overall_ctrl_median = ctrls[features].median()
    mutant_impact = (df[features] - overall_ctrl_median).abs().quantile(0.95)

    # E. Quality Score: Impact / (Total Penalty + epsilon)
    quality_score = mutant_impact / (total_penalty + 1e-6)

    # Filter out dead features and pick top 500
    variance_filter = df[features].std() > 0.01
    vetted_list = quality_score[variance_filter].sort_values(ascending=False).head(top_n).index.tolist()
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_global_stability(df_raw, feature_cols, top_n=500)

# Step 3: Perform Final Normalization (Per-Plate centering)
def robust_normalize_final(df, features):
    norm_list = []
    for plate in df['Plate'].unique():
        p_df = df[df['Plate'] == plate].copy()
        p_ctrls = p_df[p_df['Treatment'] == CONTROL_LABEL]
        if not p_ctrls.empty:
            med = p_ctrls[features].median()
            mad = (p_ctrls[features] - med).abs().median()
            p_df[features] = (p_df[features] - med) / (mad + 1e-6)
        norm_list.append(p_df)
    return pd.concat(norm_list, ignore_index=True)

df_vetted_norm = robust_normalize_final(df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols], vetted_cols)
df_vetted_norm.to_csv(os.path.join(PROJECT_ROOT, "vetted_multi_timepoint_profiles.csv"), index=False)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_multi_timepoint_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_multi_timepoint_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2","PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2","PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Setup Paths
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA"

# 2. Load Raw Data
print("Loading raw aggregated data...")
df_raw = pd.read_csv(INPUT_CSV)

# Identify features (0-6399)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_stability_only(df, features, top_n=500):
    # Aggregate Raw Control Medians
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # --- A. Within-Plate Drift (P5 T0 vs T1) ---
    # Pure technical drift within one plate
    within_plate_drift = (plate_medians.loc['PLATE5_T1'] - plate_medians.loc['PLATE5_T0']).abs()

    # --- B. Across-Plate Drift (Batch Noise per Timepoint) ---
    # Variation between different plates at the same timepoint
    timepoints = ['T0', 'T1', 'T2']
    tp_batch_noises = []

    for tp in timepoints:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            # Measure the standard deviation across plates at this timepoint
            tp_noise = plate_medians.loc[tp_plates].std(axis=0)
            tp_batch_noises.append(tp_noise)
    
    across_plate_drift = pd.concat(tp_batch_noises, axis=1).mean(axis=1)

    # --- C. Total Technical Noise ---
    # We want to minimize this value
    total_tech_noise = (within_plate_drift + across_plate_drift) / 2

    # --- D. Selection ---
    # 1. Filter out features with no variance (completely flat/dead)
    variance_filter = df[features].std() > 0.01
    
    # 2. Rank by lowest noise first
    # We sort ascending because lower noise = higher quality stability
    vetted_list = total_tech_noise[variance_filter].sort_values(ascending=True).head(top_n).index.tolist()
    
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_stability_only(df_raw, feature_cols, top_n=500)

# Step 3: Perform Final Normalization (Per-Plate centering)
def robust_normalize_final(df, features):
    norm_list = []
    for plate in df['Plate'].unique():
        p_df = df[df['Plate'] == plate].copy()
        p_ctrls = p_df[p_df['Treatment'] == CONTROL_LABEL]
        if not p_ctrls.empty:
            med = p_ctrls[features].median()
            mad = (p_ctrls[features] - med).abs().median()
            p_df[features] = (p_df[features] - med) / (mad + 1e-6)
        norm_list.append(p_df)
    return pd.concat(norm_list, ignore_index=True)

# Select metadata + the 500 most stable features
df_vetted_raw_subset = df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols]

# Apply the normalization to align the remaining minor offsets
df_vetted_norm = robust_normalize_final(df_vetted_raw_subset, vetted_cols)

# --- SAVE ---
output_filename = "vetted_stability_only_profiles.csv"
df_vetted_norm.to_csv(os.path.join(PROJECT_ROOT, output_filename), index=False)

print(f"Selection based on PURE STABILITY complete.")
print(f"Saved 500 quietest features to: {output_filename}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_stability_only_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Setup Paths
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA"

# 2. Load Raw Data
print("Loading raw aggregated data...")
df_raw = pd.read_csv(INPUT_CSV)

# Identify features (0-6399)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_stability_only(df, features, top_n=500):
    # Aggregate Raw Control Medians
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # --- A. Within-Plate Drift (P5 T0 vs T1) ---
    # Pure technical drift within one plate
    within_plate_drift = (plate_medians.loc['PLATE5_T1'] - plate_medians.loc['PLATE5_T0']).abs()

    # --- B. Across-Plate Drift (Batch Noise per Timepoint) ---
    # Variation between different plates at the same timepoint
    timepoints = ['T0', 'T1', 'T2']
    tp_batch_noises = []

    for tp in timepoints:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            # Measure the standard deviation across plates at this timepoint
            tp_noise = plate_medians.loc[tp_plates].std(axis=0)
            tp_batch_noises.append(tp_noise)
    
    across_plate_drift = pd.concat(tp_batch_noises, axis=1).mean(axis=1)

    # --- C. Total Technical Noise ---
    # We want to minimize this value
    total_tech_noise = (within_plate_drift + across_plate_drift) / 2

    # --- D. Selection ---
    # 1. Filter out features with no variance (completely flat/dead)
    variance_filter = df[features].std() > 0.01
    
    # 2. Rank by lowest noise first
    # We sort ascending because lower noise = higher quality stability
    vetted_list = total_tech_noise[variance_filter].sort_values(ascending=True).head(top_n).index.tolist()
    
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_stability_only(df_raw, feature_cols, top_n=500)



# Select metadata + the 500 most stable features
df_vetted_raw_subset = df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols]

# Apply the normalization to align the remaining minor offsets
df_vetted_norm = df_vetted_raw_subset 
# --- SAVE ---
output_filename = "vetted_stability_only_profiles.csv"
df_vetted_norm.to_csv(os.path.join(PROJECT_ROOT, output_filename), index=False)

print(f"Selection based on PURE STABILITY complete.")
print(f"Saved 500 quietest features to: {output_filename}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_stability_only_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()